In [1]:
# Import necessary libraries
import json
import os
from typing import Any, Dict

import boto3
import requests
from dotenv import load_dotenv

load_dotenv()
print("We are up and running!")

We are up and running!


AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [2]:
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

Bedrock client ready — model: eu.amazon.nova-lite-v1:0


In [3]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

I'm unable to provide real-time information as my data only goes up until 2022 and I do not have access to external sources of information that could provide the current price of Bitcoin. For the most up-to-date information, I recommend checking a reliable financial news website or using a cryptocurrency tracking app. Some popular options include CoinMarketCap, CoinGecko, and various cryptocurrency exchanges like Coinbase or Binance. Please note that the price of Bitcoin and other cryptocurrencies can be highly volatile and can change rapidly within short periods of time.


In [4]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "d52d3faa-2d13-41c8-a7a1-cecc47010737",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 12:11:01 GMT",
      "content-type": "application/json",
      "content-length": "781",
      "connection": "keep-alive",
      "x-amzn-requestid": "d52d3faa-2d13-41c8-a7a1-cecc47010737"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm unable to provide real-time information as my data only goes up until 2022 and I do not have access to external sources of information that could provide the current price of Bitcoin. For the most up-to-date information, I recommend checking a reliable financial news website or using a cryptocurrency tracking app. Some popular options include CoinMarketCap, CoinGecko, and various cryptocurrency exchanges like Coinbase or Binance. Please note that the price of Bitcoin and other cryptocurrencies can be

In [5]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

{'symbol': 'BTCUSDT', 'price': '66359.90000000'}


In [6]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [7]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

BTC Price in USDT: 66347.48


In [8]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [9]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "3e02a6e1-da37-4aa0-9b6c-493b5d4ad6dd",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 12:11:13 GMT",
      "content-type": "application/json",
      "content-length": "505",
      "connection": "keep-alive",
      "x-amzn-requestid": "3e02a6e1-da37-4aa0-9b6c-493b5d4ad6dd"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>The User has asked for the current price of Bitcoin. To get this information, I can use the \"get_crypto_price\" tool with the symbol for Bitcoin, which is BTCUSDT.</thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_osYnPZk1el3C5TJmAUDReE",
            "name": "get_crypto_price",
            "input": {
              "symbol": "BTCUSDT"
            }
          }
        }
      ]
    }
  },
  "stopReason": "tool_use",
  "usage": {
    "inputTokens": 444,


In [10]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: get_crypto_price
Tool input:     {'symbol': 'BTCUSDT'}
Tool use ID:    tooluse_osYnPZk1el3C5TJmAUDReE


In [11]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

Result from function: 66365.92


In [12]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "ad874911-1ebf-40c7-8e3e-e0ee2039f55a",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 12:11:26 GMT",
      "content-type": "application/json",
      "content-length": "430",
      "connection": "keep-alive",
      "x-amzn-requestid": "ad874911-1ebf-40c7-8e3e-e0ee2039f55a"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>I have successfully retrieved the current price of Bitcoin. The price is in USDT and is 66365.92. I can now provide this information to the User.</thinking>\n\nThe current price of Bitcoin (BTCUSDT) is 66365.92 USDT."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 546,
    "outputTokens": 66,
    "totalTokens": 612
  },
  "metrics": {
    "latencyMs": 654
  }
}


In [13]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

The current price of Bitcoin (BTCUSDT) is 66365.92 USDT.


## Example of a tool that executes any Python function

In [23]:
# Define the function
def execute_python_function(src: str) -> Any:
    """
    Executes a Python function and returns the result
    """
    return eval(src)

In [24]:
price = (execute_python_function(src="""
get_crypto_price('BTCUSDT')
"""))

price

66303.89

In [25]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "execute_python_function",
                "description": "Execute a Python function and return the result",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "src": {
                                "type": "string",
                                "description": (
                                    "The name of the function and its arguments to execute."
                                    "(e.g., get_crypto_price('BTCUSDT'), get_crypto_price('ETHUSDT'))"
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["src"],
                    }
                },
            }
        }
    ]
}

In [26]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "4f794c52-e83f-49c3-9bca-3fcee1e521ca",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 12:15:50 GMT",
      "content-type": "application/json",
      "content-length": "578",
      "connection": "keep-alive",
      "x-amzn-requestid": "4f794c52-e83f-49c3-9bca-3fcee1e521ca"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>The User wants to know the current price of Bitcoin. I can use the 'execute_python_function' tool with the 'get_crypto_price' function to retrieve the current price of Bitcoin. The symbol for Bitcoin is 'BTCUSDT'.</thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_M3psZJhCpEWtKQWWtSFOrs",
            "name": "execute_python_function",
            "input": {
              "src": "get_crypto_price('BTCUSDT')"
            }
          }
        }
      ]
   

In [27]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "994f8192-c84e-4e4f-82a1-157f851a2490",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 12:15:50 GMT",
      "content-type": "application/json",
      "content-length": "364",
      "connection": "keep-alive",
      "x-amzn-requestid": "994f8192-c84e-4e4f-82a1-157f851a2490"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>I have received the current price of Bitcoin. I can now provide this information to the User.</thinking> \n\nThe current price of Bitcoin is 66303.89."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 565,
    "outputTokens": 41,
    "totalTokens": 606
  },
  "metrics": {
    "latencyMs": 501
  }
}


In [28]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: execute_python_function
Tool input:     {'src': "get_crypto_price('BTCUSDT')"}
Tool use ID:    tooluse_M3psZJhCpEWtKQWWtSFOrs


In [29]:
# Manually call the function with the arguments the model provided
price = execute_python_function(**tool_input)
print(f"Result from function: {price}")

Result from function: 66303.9


In [30]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "ec5dc0f0-1cf6-4ae6-b183-66a1dd6ac37f",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 12:16:00 GMT",
      "content-type": "application/json",
      "content-length": "483",
      "connection": "keep-alive",
      "x-amzn-requestid": "ec5dc0f0-1cf6-4ae6-b183-66a1dd6ac37f"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>I have retrieved the current price of Bitcoin using the 'get_crypto_price' function. The current price of Bitcoin is $66,303.9.</thinking>\n\n\nThe current price of Bitcoin is $66,303.9. Please note that cryptocurrency prices are highly volatile and can change rapidly."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 582,
    "outputTokens": 78,
    "totalTokens": 660
  },
  "metrics": {
    "latencyMs": 732
  }
}


In [31]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

The current price of Bitcoin is $66,303.9. Please note that cryptocurrency prices are highly volatile and can change rapidly.
